In [1]:
import sys
#sys.path.append('/Users/theodorehuppert/VSCode/pyNIRS_toolbox/pyNIRS_toolbox')

import pyBrainAnalyzIR
import pandas as pd
import pyBrainAnalyzIR.testing

# All the processing modules are in pipelines.modules
import pyBrainAnalyzIR
import pyBrainAnalyzIR.pipelines.modules as pipelines
import pyBrainAnalyzIR.dataclasses.dataset as dataset



In [2]:
dset=dataset.DataSet()
data1,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data2,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data3,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data4,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data5,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data6,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)

dset.import_data(data1)
dset.import_data(data2)
dset.import_data(data3)
dset.import_data(data4)
dset.import_data(data5)
dset.import_data(data6)

demo=pd.DataFrame({'subject':['A','B','C','D','E','F'],'gender':['M','M','F','F','M','F'],'age':[1.,3.,5.,6.,1.,3.]})
dset.add_demographics_by_index(demo)

print("\n\nRunning Analysis:\n")
job = pipelines.events.rename_stims()
job.options['ListofChanges']={
        "1.0": "control",
        "2.0": "Tapping/Left",
        "3.0": "Tapping/Right",
        "15.0": "start marker",
    }
job = pipelines.events.remove_stims(job)
job.options['ListtoRemove']=["start marker"]
job = pipelines.preproccessing.intensity_opticaldensity(job)
#job = pipelines.filters.bandpass_filter(job)
job = pipelines.preproccessing.mbll(job)
job = pipelines.preproccessing.resample(job)
job.options['Fs']=1
job = pipelines.motion_correction.TDDR(job)
job = pipelines.glm.GLM(job)
job.options['noise_model']='ar_irls'


dset=job.run(dset)


 94%|█████████▍| 15/16 [00:00<00:00, 871.18it/s]




Running Analysis:



100%|██████████| 16/16 [00:00<00:00, 37.49it/s]


In [4]:
job = pipelines.mixedeffects.MixedEffects()
job.options['FE_formula']='Beta ~ 0 + Condition+Condition:age'
job.options['robust']=True

dset=job.run(dset)

In [5]:
dset['groupstats'].table()

,Channel,Type,Condition,Beta,StdErr,T-value,P-values,Q-values
0,S1D1,HbO,Condition[Drift 0],-3.658862,0.306538,-11.936085,6.319567e-25,5.392697e-24
1,S1D1,HbO,Condition[HRF A],0.237518,2.444463,0.097166,9.226960e-01,9.990438e-01
2,S1D1,HbO,Condition[Drift 0]:age,5.710236,0.250128,22.829250,1.309549e-56,3.352446e-55
3,S1D1,HbO,Condition[HRF A]:age,0.030171,2.098339,0.014378,9.885430e-01,9.990438e-01
4,S2D1,HbO,Condition[Drift 0],-7.614998,0.381571,-19.956971,1.045637e-48,1.673019e-47
...,...,...,...,...,...,...,...,...
123,S8D8,HbR,Condition[HRF A]:age,-0.313409,1.396479,-0.224428,8.226631e-01,9.990438e-01
124,S9D8,HbR,Condition[Drift 0],-1.614306,0.297201,-5.431699,1.676349e-07,5.646649e-07
125,S9D8,HbR,Condition[HRF A],0.197304,2.291583,0.086099,9.314772e-01,9.990438e-01
126,S9D8,HbR,Condition[Drift 0]:age,0.865383,0.153335,5.643725,5.905856e-08,2.043107e-07
